"""
================================================================================
UJI HIPOTESIS SKRIPSI - RAG QA (SQuAD v2.0)
================================================================================

DESAIN:
  Faktor A = config (4)  |  Faktor B = model (3)  |  Interaksi config x model
  Subjek repeated = query_id (query SAMA diuji di semua 12 kombinasi)

3 LANGKAH:
  (1) UJI ASUMSI    : Shapiro (normalitas) + Mauchly (sphericity)
  (2) UJI HIPOTESIS : Faithfulness -> RM-ANOVA / ART (jika asumsi gagal)
                      Abstention   -> Cochran's Q (biner)
  (3) UJI LANJUT    : Faithfulness -> pairwise Wilcoxon/t (Bonferroni)
                      Abstention   -> McNemar pairwise (Bonferroni)

================================================================================
"""

In [ ]:
!pip install pingouin statsmodels scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 7.8 MB/s eta 0:00:00


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, pingouin as pg
from scipy import stats
from itertools import combinations
from statsmodels.stats.contingency_tables import cochrans_q, mcnemar

ALPHA  = 0.05
MODELS = ["SLM", "MLM", "LLM"]
STRAT  = "B"   # <- strategi terpilih

def banner(t): print("\n"+"="*76+"\n"+t+"\n"+"="*76)
def pcol(d):
    for c in ["p-unc","p_unc"]:
        if c in d.columns: return c
def pcorr(r): return r.get("p_corr", r.get("p-corr"))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ---------- LOAD ----------
def load(strat):
    fr=[]
    for m in MODELS:
        d=pd.read_excel(f"/content/drive/MyDrive/metrik/metrics_all_configs_{m}_{strat}.xlsx","All Configs")
        d["model"]=m; fr.append(d)
    df=pd.concat(fr,ignore_index=True)
    df["config"]=df["config"].astype(str)
    df["model"]=pd.Categorical(df["model"],MODELS,ordered=True)
    return df

df = load(STRAT)
print(f"STRATEGI {STRAT} | {len(df)} baris = {df.query_id.nunique()} query x 4 config x 3 model")

STRATEGI B | 1200 baris = 100 query x 4 config x 3 model


In [ ]:
# ============================================================================
# BAGIAN A — FAITHFULNESS (kontinu, query ANSWERABLE)
# ============================================================================
banner("BAGIAN A: FAITHFULNESS")
fa = df[df.gold_is_unanswerable==False][["query_id","config","model","faithfulness"]].copy()
print(f"Subset answerable: {len(fa)} baris")



BAGIAN A: FAITHFULNESS
Subset answerable: 540 baris


In [ ]:
# ---- (1) UJI ASUMSI ----
banner("A.1 UJI ASUMSI")
normal=True
print("[a] Normalitas Shapiro-Wilk per sel config x model:")
for (c,m),g in fa.groupby(["config","model"],observed=True):
    if g["faithfulness"].nunique()<3:
        print(f"    {c} x {m}: variasi minim -> TIDAK normal"); normal=False; continue
    W,p=stats.shapiro(g["faithfulness"]); ok=p>ALPHA; normal&=ok
    print(f"    {c} x {m}: W={W:.3f} p={p:.4f} -> {'NORMAL' if ok else 'TIDAK'}")
sph=pg.sphericity(fa,dv="faithfulness",subject="query_id",within="config")
print(f"[b] Sphericity Mauchly (config): W={sph.W:.3f} p={sph.pval:.4f} -> "
      f"{'OK' if sph.spher else 'dilanggar (pakai koreksi GG)'}")
print(f"\n=> Semua sel normal? {normal}")


A.1 UJI ASUMSI
[a] Normalitas Shapiro-Wilk per sel config x model:
    config1 x SLM: variasi minim -> TIDAK normal
    config1 x MLM: W=0.698 p=0.0000 -> TIDAK
    config1 x LLM: W=0.535 p=0.0000 -> TIDAK
    config2 x SLM: W=0.801 p=0.0000 -> TIDAK
    config2 x MLM: variasi minim -> TIDAK normal
    config2 x LLM: variasi minim -> TIDAK normal
    config3 x SLM: W=0.624 p=0.0000 -> TIDAK
    config3 x MLM: W=0.702 p=0.0000 -> TIDAK
    config3 x LLM: W=0.519 p=0.0000 -> TIDAK
    config4 x SLM: W=0.700 p=0.0000 -> TIDAK
    config4 x MLM: W=0.633 p=0.0000 -> TIDAK
    config4 x LLM: variasi minim -> TIDAK normal
[b] Sphericity Mauchly (config): W=0.493 p=0.0000 -> dilanggar (pakai koreksi GG)

=> Semua sel normal? False


In [ ]:
# ---- (2) UJI HIPOTESIS ----
banner("A.2 UJI HIPOTESIS (config, model, interaksi)")
use_param=normal
print(f"Metode: {'RM-ANOVA 2-faktor' if use_param else 'ART RM-ANOVA 2-faktor (NON-parametrik, krn normalitas gagal)'}\n")

def rm2(data,dv): return pg.rm_anova(data=data,dv=dv,within=["config","model"],
                                     subject="query_id",detailed=True)
if use_param:
    aov=rm2(fa,"faithfulness"); P=pcol(aov)
    eff=[(r["Source"],r["F"],r[P]) for _,r in aov.iterrows() if r["Source"]!="Error"]
else:
    g=fa.copy(); g["grand"]=g["faithfulness"].mean()
    mA=g.groupby("config",observed=True)["faithfulness"].transform("mean")
    mB=g.groupby("model",observed=True)["faithfulness"].transform("mean")
    mAB=g.groupby(["config","model"],observed=True)["faithfulness"].transform("mean")
    g["c"]=g["faithfulness"]-(mB-g["grand"])-(mAB-mA-mB+g["grand"])
    g["m"]=g["faithfulness"]-(mA-g["grand"])-(mAB-mA-mB+g["grand"])
    g["i"]=g["faithfulness"]-(mA-g["grand"])-(mB-g["grand"])
    eff=[]
    for name,col,src in [("config","c","config"),("model","m","model"),
                         ("config * model","i","config * model")]:
        a=rm2(g.assign(faithfulness=g[col].rank()),"faithfulness"); Pp=pcol(a)
        r=a[a["Source"]==src].iloc[0]; eff.append((name,r["F"],r[Pp]))

sig={}
print(f"{'Efek':18s}{'F':>10s}{'p':>14s}   Keputusan")
for name,F,p in eff:
    s=p<ALPHA; sig[name]=s
    print(f"{name:18s}{F:10.2f}{p:14.3e}   {'SIGNIFIKAN' if s else 'tidak'}")


A.2 UJI HIPOTESIS (config, model, interaksi)
Metode: ART RM-ANOVA 2-faktor (NON-parametrik, krn normalitas gagal)

Efek                       F             p   Keputusan
config                 17.05     2.044e-09   SIGNIFIKAN
model                  30.20     1.033e-10   SIGNIFIKAN
config * model          9.38     2.453e-09   SIGNIFIKAN


In [ ]:
# ---- (3) UJI LANJUT ----
banner("A.3 UJI LANJUT (POST-HOC)")
if sig.get("config"):
    print(">>> Pairwise CONFIG (Bonferroni):")
    ph=pg.pairwise_tests(data=fa,dv="faithfulness",within="config",
                         subject="query_id",parametric=use_param,padjust="bonf")
    for _,r in ph.iterrows():
        print(f"   {r['A']} vs {r['B']}: p_corr={pcorr(r):.4f}  hedges={r['hedges']:.3f}  "
              f"{'SIG' if pcorr(r)<ALPHA else 'ns'}")
if sig.get("model"):
    print("\n>>> Pairwise MODEL (Bonferroni):")
    ph=pg.pairwise_tests(data=fa,dv="faithfulness",within="model",
                         subject="query_id",parametric=use_param,padjust="bonf")
    for _,r in ph.iterrows():
        print(f"   {r['A']} vs {r['B']}: p_corr={pcorr(r):.4f}  hedges={r['hedges']:.3f}  "
              f"{'SIG' if pcorr(r)<ALPHA else 'ns'}")

print("\n[Mean Faithfulness answerable per config x model]")
print(fa.groupby(["config","model"],observed=True)["faithfulness"].mean().unstack().round(3).to_string())



A.3 UJI LANJUT (POST-HOC)
>>> Pairwise CONFIG (Bonferroni):
   config1 vs config2: p_corr=0.0770  hedges=0.451  ns
   config1 vs config3: p_corr=0.0001  hedges=0.825  SIG
   config1 vs config4: p_corr=0.0027  hedges=0.713  SIG
   config2 vs config3: p_corr=0.1028  hedges=0.370  ns
   config2 vs config4: p_corr=0.0838  hedges=0.254  ns
   config3 vs config4: p_corr=1.0000  hedges=-0.119  ns

>>> Pairwise MODEL (Bonferroni):
   SLM vs MLM: p_corr=0.0064  hedges=-0.631  SIG
   SLM vs LLM: p_corr=0.0000  hedges=-1.358  SIG
   MLM vs LLM: p_corr=0.0002  hedges=-0.613  SIG

[Mean Faithfulness answerable per config x model]
model      SLM    MLM    LLM
config                      
config1  0.778  0.776  0.880
config2  0.539  0.644  0.889
config3  0.333  0.626  0.789
config4  0.309  0.678  0.867


In [ ]:
# ============================================================================
# BAGIAN B — ABSTENTION (biner, semua query)
# ============================================================================
banner("BAGIAN B: ABSTENTION — Cochran's Q + McNemar (per model)")
for m in MODELS:
    ab=df[df.model==m].pivot_table(index="query_id",columns="config",
                                   values="abstention_correct").dropna().astype(int)
    cfgs=sorted(ab.columns)
    print(f"\n--- MODEL {m} ---")
    print("  Proporsi benar: "+" | ".join(f"{c}={ab[c].mean():.3f}" for c in cfgs))
    try:
        q=cochrans_q(ab.values); ok=not np.isnan(q.pvalue)
    except Exception: ok=False
    if not ok:
        print("  Cochran Q tak terdefinisi (proporsi konstan) -> tidak ada efek config."); continue
    sg=q.pvalue<ALPHA
    print(f"  Cochran Q={q.statistic:.3f} df={len(cfgs)-1} p={q.pvalue:.4f} -> {'SIGNIFIKAN' if sg else 'tidak'}")
    if sg:
        pairs=list(combinations(cfgs,2)); k=len(pairs)
        for a,b in pairs:
            tb=pd.crosstab(ab[a],ab[b]).reindex(index=[0,1],columns=[0,1],fill_value=0)
            r=mcnemar(tb.values,exact=True); pb=min(r.pvalue*k,1.0)
            print(f"    {a} vs {b}: p_bonf={pb:.4f} {'SIG' if pb<ALPHA else 'ns'}")



BAGIAN B: ABSTENTION — Cochran's Q + McNemar (per model)

--- MODEL SLM ---
  Proporsi benar: config1=0.430 | config2=0.490 | config3=0.600 | config4=0.540
  Cochran Q=7.536 df=3 p=0.0566 -> tidak

--- MODEL MLM ---
  Proporsi benar: config1=0.490 | config2=0.430 | config3=0.470 | config4=0.500
  Cochran Q=1.605 df=3 p=0.6583 -> tidak

--- MODEL LLM ---
  Proporsi benar: config1=0.520 | config2=0.600 | config3=0.530 | config4=0.640
  Cochran Q=6.270 df=3 p=0.0992 -> tidak


In [ ]:
import os
os.listdir("/content/drive/MyDrive/metrik")  # cek apakah folder shared muncul

['metrics_all_configs_SLM_B.xlsx',
 'metrics_all_configs_MLM_B.xlsx',
 'metrics_all_configs_LLM_B.xlsx',
 'metrics_all_configs_LLM_A.xlsx',
 'metrics_all_configs_MLM_A.xlsx',
 'metrics_all_configs_SLM_A.xlsx']

In [ ]:
PATH = {
    "A": "/content/drive/MyDrive/metrik",
    "B": "/content/drive/MyDrive/metrik",
}

def load(strat):
    fr=[]
    for m in MODELS:
        d=pd.read_excel(f"{PATH[strat]}/metrics_all_configs_{m}_{strat}.xlsx","All Configs")
        d["model"]=m; fr.append(d)
    df=pd.concat(fr,ignore_index=True)
    df["config"]=df["config"].astype(str)
    df["model"]=pd.Categorical(df["model"],MODELS,ordered=True)
    return df

In [ ]:
# ============================================================================
# LAMPIRAN — PERBANDINGAN STRATEGI A vs B
# ============================================================================
banner("LAMPIRAN: PERBANDINGAN STRATEGI A vs B (rata-rata keseluruhan)")
rows=[]
for s in ["A","B"]:
    d=load(s); ans=d[d.gold_is_unanswerable==False]
    rows.append(dict(Strategi=s,
        Faithfulness_answerable=round(ans.faithfulness.mean(),4),
        Abstention=round(d.abstention_correct.mean(),4),
        EM=round(d.em.mean(),4), F1_token=round(d.f1_token.mean(),4)))
print(pd.DataFrame(rows).set_index("Strategi").to_string())
print("\nCatatan: Strategi A memiliki anomali (Faithfulness MLM = 0.000 di semua")
print("config). Strategi B tidak. -> Strategi B dipilih sebagai eksperimen utama.")

banner("SELESAI")


LAMPIRAN: PERBANDINGAN STRATEGI A vs B (rata-rata keseluruhan)
          Faithfulness_answerable  Abstention      EM  F1_token
Strategi                                                       
A                          0.4259       0.525  0.1942    0.2356
B                          0.6756       0.520  0.0408    0.1102

Catatan: Strategi A memiliki anomali (Faithfulness MLM = 0.000 di semua
config). Strategi B tidak. -> Strategi B dipilih sebagai eksperimen utama.

SELESAI
